In [4]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def f(x):
    return 3 * x**2 - 4*x + 5

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)
plt.xlabel('x')
plt.ylabel('f(x)')
plt.show()

In [ ]:
h = 0.000001
x = 2/3
(f(x+h) - f(x)) /h

In [ ]:
a = 2
b = -3
c = 10
d = a * b +c
d

In [ ]:
h = 0.0001
a = 2
b = -3
c = 10
d1 = a * b + c
c +=h
d2 = a * b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2 - d1) / h)

In [1]:
class Value:
    def __init__(self, data, _chirldren=(), _op='', label=''):
        # self.data = data
        # Value.data should be a plain scalar (float/int).
        # If a Value accidentally gets wrapped inside another Value, unwrap it.
        if isinstance(data, Value):
            data = data.data
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_chirldren)
        self._op = _op
        self.label = label
    
    def __repr__(self):
        return f'Value({self.data})'
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)    # To make Value(X) + 1 being work 
        out = Value(self.data + other.data, _chirldren=(self, other), _op='+')

        def _backward():
            self.grad += 1.0 * out.grad         # a.grad (pointing to a). += is required; otherwise there will be a bug (e.g. a + a -> grad = 1)
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out
    
    def __radd__(self, other):      #make swap of variable being available 
        return self + other
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other + (-self)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)    # To make Value(X) + 1 being work 
        out = Value(self.data * other.data, _chirldren=(self, other), _op='*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __rmul__(self, other):
        return self * other    

    def __pow__(self, other):
        assert isinstance(other, (int, float))      # only for int and float
        out = Value(self.data ** other, _chirldren=(self,), _op=f'**{other}')

        def _backward():
            self.grad += other * self.data**(other-1) * out.grad
        out._backward = _backward

        return out
    
    def __truediv__(self, other):
        return self * other**-1
    
    def tanh(self):
        n = self.data
        #t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        t = math.tanh(n)
        out = Value(t, (self,), _op='tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), _op='exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


# a = Value(2.0, label = 'a')
# b = Value(-3.0, label = 'b')
# c = Value(10.0, label = 'c')
# e = a * b; e.label = 'e'
# d = e + c; d.label = 'd'   # (a__mul__ b) __add__ c
# f = Value(-2.0, label = 'f')
# L = f * d; L.label = 'L'
# d
#d._prev

# L.grad = 1.0
# f.grad = 4.0
# d.grad = -2.0
# c.grad = -2.0
# e.grad = -2.0
# a.grad = -2.0 * -3.0
# b.grad = -2.0 * 2.0

# a.data += h * a.grad
# b.data += h * b.grad
# c.data += h * c.grad
# f.data += h * f.grad

# e = a * b
# d = e + c
# L = f * d

# print(L.data)
#draw_dot(L)

In [ ]:
a = Value(2)
b = Value(4)
a / b

In [2]:
from graphviz import Digraph
def trace(root):
    # builds a set of all nodes and edges in the computational graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    
    return nodes, edges

def draw_dot(root):
    dat = Digraph(format='svg', graph_attr={'rankdir': 'LR'})  # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dat.node(name=uid, label="{%s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if any value in the graph, create a rectanglar ('record') node for it
            dat.node(name=uid + n._op, label=n._op)
            # and connect this node to it
            dat.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dat.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dat

In [ ]:
def lol():

    h = 0.0001

    a = Value(2.0, label = 'a')
    b = Value(-3.0, label = 'b')
    c = Value(10.0, label = 'c')
    e = a * b; e.label = 'e'
    d = e + c; d.label = 'd'   # (a__mul__ b) __add__ c
    f = Value(-2.0, label = 'f')
    L = f * d; L.label = 'L'
    L1 = L.data

    a = Value(2.0, label = 'a')
    b = Value(-3.0, label = 'b')
    b.data += h
    c = Value(10.0, label = 'c')
    e = a * b; e.label = 'e'
    d = e + c; d.label = 'd'   # (a__mul__ b) __add__ c
    f = Value(-2.0, label = 'f')
    L = f * d; L.label = 'L'
    L2 = L.data

    print((L2 - L1) / h)

lol()

In [ ]:
plt.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2))); plt.grid()

In [ ]:
# input x1, x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

#weights w1, w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

#bias of the neuron
b = Value(6.8813735870195432, label='b')

#x1w1 + x2w2 + b
x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [ ]:
# o.grad = 1.0
# n.grad = (1 - o.data ** 2) * o.grad
# x1w1x2w2.grad = n.grad
# b.grad = n.grad
# x1w1.grad = n.grad
# x2w2.grad = n.grad
# x2.grad = w2.data * x2w2.grad
# w2.grad = x2.data * x2w2.grad
# x1.grad = w1.data * x1w1.grad
# w1.grad = x1.data * x1w1.grad

# o.grad = 1.0
# o._backward()
# n._backward()
# b._backward()
# x1w1x2w2._backward()
# x1w1._backward()
# x2w2._backward()

o.backward()
draw_dot(o)


In [ ]:
o.grad = 1.0

topo = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)
build_topo(o)
topo

for node in reversed(topo):
    node._backward()

In [ ]:
# Remember that:
a = Value(3.0, label='a')
b = a + a; b.label = 'b'        # same symbol in the same line
b.backward()
draw_dot(b)



In [ ]:
# input x1, x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

#weights w1, w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

#bias of the neuron
b = Value(6.8813735870195432, label='b')

#x1w1 + x2w2 + b
x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'

e = (2*n).exp()
o = (e - 1) / (e + 1); o.label = 'o'
#o = n.tanh(); o.label = 'o'
o.backward()
draw_dot(o)

In [ ]:
# torch version
import torch
x1 = torch.tensor([2.0]).double()                   ; x1.requires_grad_(True)
x2 = torch.tensor([0.0]).double()                   ; x2.requires_grad_(True)
w1 = torch.tensor([-3.0]).double()                  ; w1.requires_grad_(True)
w2 = torch.tensor([1.0]).double()                   ; w2.requires_grad_(True)
b = torch.tensor([6.8813735870195432]).double()     ; b.requires_grad_(True)
n = x1 * w1 + x2 * w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()            # this is the function in torch, not defined by us

print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

In [5]:
import random
class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    
    def __call__(self, x):
        # w * x + b
        #print(list(zip(self.w, x)))
        act = sum((wi * xi for wi, xi in zip(self.w, x))) + self.b
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs      # no sqaure bracket output if there is only one neuron in the layer 
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
        # params = []
        # for neuron in self.neurons:
        #     params.extend(neuron.parameters())
        # return params

class MLP:                  # serier of layer

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

x = [2.0, 3.0, -1.0]
#n = Layer(2, 3)
n = MLP(3, [4, 4, 1])   # 3 input numbers. 4 neurons in the first layer, 4 neurons in the second layer, and 1 neuron in the third layer
n(x)

Value(0.836336499066561)

In [54]:
print(n.parameters())
print(len(n.parameters()))

[Value(-0.6685798500539342), Value(0.9353306025981363), Value(-0.09951619850106708), Value(0.001500482357484234), Value(0.971481520736521), Value(-0.8018661982469439), Value(-0.5236972747404436), Value(0.9388262046210434), Value(0.12964764917973315), Value(-0.34298478028717816), Value(0.7251231856806841), Value(0.1177762981019479), Value(-0.033358651416073304), Value(-0.2507356465579047), Value(0.31240647365634033), Value(0.41950977558924385), Value(0.24649899121188001), Value(-0.5745309872255258), Value(0.9985911978218216), Value(-0.583441050012462), Value(-0.1541680426294827), Value(0.04814804135082862), Value(0.2419008702694303), Value(0.30989976517925344), Value(-0.5338628490938024), Value(-0.1247190975302388), Value(-0.5336758488769509), Value(-0.017947598248072794), Value(0.235026031017054), Value(-0.6415589686266403), Value(0.34201037758529695), Value(0.44311408343659386), Value(-0.8328257761836007), Value(-0.6227810435969872), Value(0.70107062496314), Value(-0.5575643414464329)

In [ ]:
draw_dot(n(x))

In [55]:
xs = [[2.0, 3.0, -1.0], 
      [3.0, -1.0, 0.5], 
      [0.5, 1.0, 1.0],
      [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets
ypred = [n(x) for x in xs]
ypred

[Value(0.7415139723048868),
 Value(0.9038392425491992),
 Value(0.5921621734032428),
 Value(0.8545915259910043)]

In [56]:
# how the make ypred close to ys? Root mean square error loss
loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])
loss

Value(6.247542898713671)

In [58]:
loss.backward()
print(n.layers[0].neurons[0].w[0].grad)
print(n.layers[0].neurons[0].w[0].data)

-10.536045604821586
-0.6685798500539342


In [ ]:
print(n.layers[0].neurons[0].w[0].data)
print(n.layers[0].neurons[0].w[0].grad)
# With looping the loss backward and parameters update, we can see the loss is decreasing
loss.backward()
for p in n.parameters():            
    p.data += -0.01 * p.grad        # 0.01 is the learning rate
ypred = [n(x) for x in xs]
print(ypred)
loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])
print(loss)

1.6486315905323108
-9.595161850218144
[Value(0.9989740590776364), Value(-0.9999798649368848), Value(-0.9999798623603156), Value(0.9989701304944705)]
Value(2.11399691989868e-06)


In [ ]:
draw_dot(loss)

In [7]:
xs = [[2.0, 3.0, -1.0], 
      [3.0, -1.0, 0.5], 
      [0.5, 1.0, 1.0],
      [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

for k in range(2000):
    
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum([(yout - ygt)**2 for ygt, yout in zip(ys, ypred)])
    
    # backward pass (zero grads, then backprop)
    for p in n.parameters():        # Remember the def Value is accumulation; the previous gradient will add on it if p is not reset.
        p.grad = 0.0
    loss.backward()
    
    for p in n.parameters():
        p.data += -0.01 * p.grad
    
    print(k, loss.data)

0 0.00139214008471399
1 0.0013913957565886055
2 0.001390652209469284
3 0.001389909442137082
4 0.001389167453375599
5 0.0013884262419709675
6 0.001387685806711806
7 0.0013869461463892475
8 0.0013862072597969235
9 0.0013854691457309493
10 0.0013847318029899303
11 0.0013839952303749535
12 0.0013832594266895667
13 0.0013825243907398077
14 0.001381790121334141
15 0.001381056617283505
16 0.001380323877401284
17 0.0013795919005032938
18 0.001378860685407784
19 0.0013781302309354528
20 0.001377400535909389
21 0.0013766715991551228
22 0.0013759434195005792
23 0.0013752159957760948
24 0.0013744893268143941
25 0.0013737634114506065
26 0.0013730382485222332
27 0.0013723138368691495
28 0.001371590175333628
29 0.0013708672627602875
30 0.001370145097996114
31 0.0013694236798904406
32 0.0013687030072949602
33 0.0013679830790637058
34 0.001367263894053037
35 0.001366545451121665
36 0.0013658277491305972
37 0.0013651107869431904
38 0.001364394563425094
39 0.0013636790774442698
40 0.001362964327870979
41

In [8]:
print(ypred)

[Value(0.9909092236507862), Value(-0.9908301585953962), Value(-0.9846875827690154), Value(0.9836703139306324)]


In [ ]:
# Homework 1 (python script, .py file): build a 6-layer MLP with 16 neurons in each layer, and train it on the above dataset. You should be able to get zero loss (or very close to zero loss) after training. [micrograd]

# Homework 2: same as Homework 1, but using PyTorch. [PyTorch]